In [1]:
import numpy as np
import matplotlib.pyplot as plt 

'2026-03-07T03:02:29.910' TS 
'2026-03-07T03:03:41.140' TS 

'2026-03-07T03:02:54.399' TN

In [62]:
# -----------------------------------------------------------------------------
# OBSERVATORY COORDINATES (latitude, longitude in degrees)
# -----------------------------------------------------------------------------
# North observatory (e.g. Morocco)
lat_N  =  31.2061
lon_N  =  -7.8664

# South observatory (e.g. Chile)
lat_S  = -29.2546
lon_S  = -70.7394

# -----------------------------------------------------------------------------
# ASTEROID OBSERVATIONS
# -----------------------------------------------------------------------------
def hms_to_deg(h, m, s):
    """Hours Minutes Seconds → Degrees (for RA)"""
    return (h + m/60 + s/3600) * 15

def dms_to_deg(d, m, s):
    """Degrees Minutes Seconds → Degrees (for Dec)"""
    sign = -1 if d < 0 else 1
    return sign * (abs(d) + m/60 + s/3600)

# Position from North observatory
ra_N  = hms_to_deg(12,  5, 23.0747)   # → degrees
dec_N = dms_to_deg(-2, 43,  7.849)    # → degrees
t_N   = 24.489                         # seconds

# Position from South observatory
ra_S  = hms_to_deg(12, 5, 25.7487)   # → degrees
dec_S = dms_to_deg(-2, 42, 24.382)    # → degrees
t_S   = 0.0                            # seconds

# -----------------------------------------------------------------------------
# PROPER MOTION of the asteroid (from a sequence of images, same observatory)
# Units: degrees per second
# -----------------------------------------------------------------------------
# fit from multiple positions on the same night
x_S1, y_S1 = 504.069, 553.947 # '2026-03-07T03:02:29.910' TS
x_S2, y_S2 = 504.456, 554.722 # '2026-03-07T03:03:41.140' TS

dt_12 = 3*60 + 41.140 - (2*60 + 29.910) # seconds
dt_S_N = 54.399 - 29.910 # seconds

times_seq = np.array([0, dt_12])        # seconds
ras_seq   = np.array([ra_S, hms_to_deg(12, 5, 25.5080)])          # degrees
decs_seq  = np.array([dec_S, dms_to_deg(-2, 42, 22.0747)])            # degrees

ra_rate  = np.polyfit(times_seq, ras_seq,  1)[0]     # deg/sec
dec_rate = np.polyfit(times_seq, decs_seq, 1)[0]     # deg/sec

print("=" * 60)
print("  PARALLAX DISTANCE CALCULATOR — Near-Earth Asteroid")
print("=" * 60)

print(f"\n[1] PROPER MOTION")
print(f"    RA  rate : {ra_rate  * 3600:.5f} arcsec/sec")
print(f"    Dec rate : {dec_rate * 3600:.5f} arcsec/sec")

R_earth_km = 6371.0
R_earth_au = R_earth_km / 149597870.700    # 1 AU in km

def to_xyz(lat_deg, lon_deg, R=1.0):
    """Convert geographic coordinates to 3D unit vector."""
    lat = np.radians(lat_deg)
    lon = np.radians(lon_deg)
    return R * np.array([
        np.cos(lat) * np.cos(lon),
        np.cos(lat) * np.sin(lon),
        np.sin(lat)
    ])

p_N = to_xyz(lat_N, lon_N, R_earth_km)
p_S = to_xyz(lat_S, lon_S, R_earth_km)

baseline_km = np.linalg.norm(p_S - p_N)
baseline_au = baseline_km / 1.496e8

print(f"\n[2] BASELINE")
print(f"    North observatory : lat={lat_N}°  lon={lon_N}°")
print(f"    South observatory : lat={lat_S}°  lon={lon_S}°")
print(f"    Chord distance    : {baseline_km:.1f} km  =  {baseline_au:.8f} AU")

dt = t_S - t_N   # seconds

ra_S_corr  = ra_S  - ra_rate  * dt
dec_S_corr = dec_S - dec_rate * dt

print(f"\n[3] PROPER MOTION CORRECTION  (dt = {dt:.1f} s)")
print(f"    Original  South RA  : {ra_S:.6f}°")
print(f"    Corrected South RA  : {ra_S_corr:.6f}°")
print(f"    Original  South Dec : {dec_S:.6f}°")
print(f"    Corrected South Dec : {dec_S_corr:.6f}°")

mean_dec = np.radians((dec_N + dec_S_corr) / 2)

delta_ra_arcsec  = (ra_S_corr - ra_N) * np.cos(mean_dec) * 3600
delta_dec_arcsec = (dec_S_corr - dec_N) * 3600

parallax_arcsec = np.sqrt(delta_ra_arcsec**2 + delta_dec_arcsec**2)
parallax_rad    = np.radians(parallax_arcsec / 3600)

print(f"\n[4] PARALLAX ANGLE")
print(f"    ΔRA  : {delta_ra_arcsec:.4f} arcsec")
print(f"    ΔDec : {delta_dec_arcsec:.4f} arcsec")
print(f"    Total parallax : {parallax_arcsec:.4f} arcsec")

distance_au = baseline_au / np.tan(parallax_rad)
distance_km = distance_au * 1.496e8

print(f"\n[5] DISTANCE TO ASTEROID")
print(f"    Distance : {distance_au:.5f} AU")
print(f"    Distance : {distance_km:.0f} km")
print(f"    Distance : {distance_km/384400:.2f} lunar distances")



  PARALLAX DISTANCE CALCULATOR — Near-Earth Asteroid

[1] PROPER MOTION
    RA  rate : -0.05069 arcsec/sec
    Dec rate : 0.03239 arcsec/sec

[2] BASELINE
    North observatory : lat=31.2061°  lon=-7.8664°
    South observatory : lat=-29.2546°  lon=-70.7394°
    Chord distance    : 8608.9 km  =  0.00005755 AU

[3] PROPER MOTION CORRECTION  (dt = -24.5 s)
    Original  South RA  : 181.357286°
    Corrected South RA  : 181.356941°
    Original  South Dec : -2.706773°
    Corrected South Dec : -2.706552°

[4] PARALLAX ANGLE
    ΔRA  : 38.8251 arcsec
    ΔDec : 44.2603 arcsec
    Total parallax : 58.8758 arcsec

[5] DISTANCE TO ASTEROID
    Distance : 0.20161 AU
    Distance : 30160202 km
    Distance : 78.46 lunar distances
